
# Amazon Search Ranking — Week 6 V1
## Cold-Start Subset Definition + Baseline Benchmark

**Goal:** isolate products with zero/minimal historical interaction and evaluate the existing ranking signals specifically on this cold-start subset.

### Recommended uploads
Upload the ZIPs you already produced in earlier weeks. The notebook is designed to auto-detect useful pair-level files inside them.

Recommended:
- `amazon_week5_v1_feature_dataset.zip`
- `amazon_week5_v2_lambdamart_results.zip`
- `amazon_week4_v1_sbert_faiss_results.zip`
- `amazon_week4_v2_clip_multimodal_results.zip`
- `amazon_week2_baseline_ranking_results.zip`

You may upload fewer files if your Week 5 feature dataset already contains BM25 / SBERT / CLIP / LambdaMART scores.

### V1 outputs
The notebook creates:
- `cold_start_subset.csv`
- `cold_start_definition.json`
- `cold_start_benchmark_metrics.csv`
- `cold_start_per_query_metrics.csv`
- `historical_benchmark_metrics.csv`
- `cold_start_summary.txt`
- diagnostic plots
- `amazon_week6_v1_coldstart_benchmark_results.zip`

### Important methodological rule
1. If real historical count columns exist (review/click/purchase/popularity/etc.), V1 uses them.
2. If no real historical signal exists, V1 **does not pretend** that relevance labels are historical behavior. Instead it creates a deterministic **simulated unseen-product holdout** and clearly records that choice in the report.


In [ ]:

# =========================
# 0. INSTALL + IMPORTS
# =========================
!pip -q install pyarrow openpyxl

import os, re, json, math, hashlib, shutil, zipfile, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path("/content/week6_v1")
UPLOAD_DIR = ROOT / "uploads"
EXTRACT_DIR = ROOT / "extracted"
OUT_DIR = ROOT / "outputs"

for p in [ROOT, UPLOAD_DIR, EXTRACT_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Main knobs
K = 10
MIN_QUERIES_FOR_REPORT = 1
COLD_START_POSITIVE_QUANTILE = 0.20
SIMULATED_COLD_START_FRACTION = 0.20
RANDOM_SEED = 42

print("Week 6 V1 environment ready.")


In [ ]:

# =========================
# 1. UPLOAD PREVIOUS-WEEK ZIP/CSV/PARQUET FILES
# =========================
from google.colab import files

uploaded = files.upload()

for name, content in uploaded.items():
    target = UPLOAD_DIR / name
    target.write_bytes(content)

print("\nUploaded:")
for p in sorted(UPLOAD_DIR.iterdir()):
    print(" -", p.name)


In [ ]:

# =========================
# 2. EXTRACT ZIP FILES
# =========================
for p in UPLOAD_DIR.iterdir():
    if p.suffix.lower() == ".zip":
        dest = EXTRACT_DIR / p.stem
        dest.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(p, "r") as z:
                z.extractall(dest)
            print(f"Extracted: {p.name}")
        except Exception as e:
            print(f"[WARN] Could not extract {p.name}: {e}")
    else:
        shutil.copy2(p, EXTRACT_DIR / p.name)

all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]
print(f"\nTotal extracted files: {len(all_files)}")
for p in all_files[:80]:
    print(" -", p.relative_to(EXTRACT_DIR))


In [ ]:

# =========================
# 3. LOAD ALL TABULAR FILES
# =========================
def read_table(path):
    ext = path.suffix.lower()
    try:
        if ext == ".csv":
            return pd.read_csv(path)
        if ext in [".parquet", ".pq"]:
            return pd.read_parquet(path)
        if ext in [".xlsx", ".xls"]:
            return pd.read_excel(path)
        if ext in [".jsonl", ".ndjson"]:
            return pd.read_json(path, lines=True)
        if ext == ".json":
            obj = json.loads(path.read_text(errors="ignore"))
            if isinstance(obj, list):
                return pd.DataFrame(obj)
            if isinstance(obj, dict):
                # only accept record-like dicts
                for key in ["data", "records", "rows", "results"]:
                    if key in obj and isinstance(obj[key], list):
                        return pd.DataFrame(obj[key])
        return None
    except Exception as e:
        print(f"[WARN] Failed reading {path.name}: {e}")
        return None

def normalize_col(c):
    c = str(c).strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    return c.strip("_")

tables = []
for p in all_files:
    if p.suffix.lower() not in [".csv", ".parquet", ".pq", ".xlsx", ".xls", ".json", ".jsonl", ".ndjson"]:
        continue
    df = read_table(p)
    if df is None or df.empty:
        continue
    df = df.copy()
    df.columns = [normalize_col(c) for c in df.columns]
    source = str(p.relative_to(EXTRACT_DIR))
    tables.append({"source": source, "path": p, "df": df})

print(f"Loaded {len(tables)} non-empty tables.\n")
for i, t in enumerate(tables):
    print(f"[{i}] {t['source']}")
    print(f"    shape={t['df'].shape}")
    print(f"    columns={list(t['df'].columns)[:30]}")


In [ ]:

# =========================
# 4. SCHEMA DETECTION
# =========================
QUERY_KEYS = [
    "query_id", "qid", "query", "normalized_query", "clean_query",
    "search_query", "query_text"
]
PRODUCT_KEYS = [
    "product_id", "asin", "parent_asin", "item_id", "doc_id",
    "sku", "product_asin"
]
LABEL_KEYS = [
    "relevance", "relevance_label", "label", "target", "y_true",
    "proxy_relevance", "relevance_score_label", "judgment", "grade"
]

HISTORY_EXACT = {
    "click_count", "clicks", "num_clicks",
    "review_count", "reviews", "num_reviews", "rating_count", "ratings_count",
    "purchase_count", "purchases", "num_purchases", "order_count",
    "sales_count", "interaction_count", "interactions",
    "impression_count", "impressions",
    "view_count", "views",
    "popularity", "popularity_count"
}

def find_first(cols, candidates):
    cols = set(cols)
    for c in candidates:
        if c in cols:
            return c
    return None

def score_family(col):
    c = col.lower()
    if "bm25" in c and "rank" not in c:
        return "BM25"
    if ("sbert" in c or "sentence_bert" in c or "semantic_score" in c) and "rank" not in c:
        return "SBERT"
    if "clip" in c and "rank" not in c:
        return "CLIP"
    if ("lambda" in c or "lambdamart" in c or "ltr_score" in c or "lightgbm" in c) and "rank" not in c:
        return "LambdaMART"
    if "listnet" in c and "rank" not in c:
        return "ListNet"
    return None

def detect_schema(df):
    q = find_first(df.columns, QUERY_KEYS)
    p = find_first(df.columns, PRODUCT_KEYS)
    y = find_first(df.columns, LABEL_KEYS)
    history = [c for c in df.columns if c in HISTORY_EXACT]
    scores = [(c, score_family(c)) for c in df.columns if score_family(c)]
    return {"query": q, "product": p, "label": y, "history": history, "scores": scores}

schemas = []
for i, t in enumerate(tables):
    s = detect_schema(t["df"])
    schemas.append(s)
    print(f"[{i}] {t['source']}")
    print("    query   :", s["query"])
    print("    product :", s["product"])
    print("    label   :", s["label"])
    print("    history :", s["history"])
    print("    scores  :", s["scores"][:20])


In [ ]:

# =========================
# 5. BUILD A MASTER QUERY-PRODUCT TABLE
# =========================
# Select the best base table:
# - must have query + product
# - strongly prefer relevance label
# - prefer more rows and more ranking score columns

candidates = []
for i, (t, s) in enumerate(zip(tables, schemas)):
    if s["query"] and s["product"]:
        quality = (
            1000000 * int(s["label"] is not None)
            + 10000 * len(s["scores"])
            + len(t["df"])
        )
        candidates.append((quality, i))

if not candidates:
    raise RuntimeError(
        "No pair-level table with both query and product IDs/text was detected. "
        "Upload the Week 5 feature dataset or another query-product result file."
    )

_, base_i = max(candidates)
base_t = tables[base_i]
base_s = schemas[base_i]
master = base_t["df"].copy()

# Canonical keys
master = master.rename(columns={
    base_s["query"]: "__query__",
    base_s["product"]: "__product__",
})
if base_s["label"]:
    master = master.rename(columns={base_s["label"]: "__label__"})

print("Base table:", base_t["source"], master.shape)

# Add useful columns from other pair-level tables
for i, (t, s) in enumerate(zip(tables, schemas)):
    if i == base_i or not s["query"] or not s["product"]:
        continue

    other = t["df"].copy()
    other = other.rename(columns={s["query"]: "__query__", s["product"]: "__product__"})

    keep = ["__query__", "__product__"]
    if "__label__" not in master.columns and s["label"]:
        other = other.rename(columns={s["label"]: "__label__"})
        keep.append("__label__")

    useful = []
    for c in other.columns:
        if c in keep:
            continue
        if c in HISTORY_EXACT or score_family(c):
            useful.append(c)

    # Keep only useful columns not already present; rename collisions by source index
    useful = [c for c in useful if c not in master.columns]
    if not useful:
        continue

    sub = other[keep + useful].copy()
    sub = sub.drop_duplicates(["__query__", "__product__"])
    master = master.merge(sub, on=["__query__", "__product__"], how="left")
    print(f"Merged {len(useful)} columns from:", t["source"])

# Numeric cleanup
for c in master.columns:
    if c.startswith("__"):
        continue
    if master[c].dtype == "object":
        converted = pd.to_numeric(master[c], errors="coerce")
        if converted.notna().mean() > 0.80:
            master[c] = converted

print("\nMASTER:", master.shape)
print(master.head())
print("\nColumns:")
print(list(master.columns))


In [ ]:

# =========================
# 6. FIND / CREATE RELEVANCE LABEL
# =========================
if "__label__" not in master.columns:
    # Last chance: locate a likely numeric relevance column in the base table
    fuzzy = [
        c for c in master.columns
        if any(k in c for k in ["relev", "label", "judg", "grade", "target"])
        and c not in ["__query__", "__product__"]
    ]
    if fuzzy:
        candidate = fuzzy[0]
        master["__label__"] = pd.to_numeric(master[candidate], errors="coerce")
        print("Using inferred relevance label:", candidate)
    else:
        raise RuntimeError(
            "No relevance label was found. V1 needs query-product relevance labels "
            "to compute NDCG@10 / Recall@10 / MRR."
        )

master["__label__"] = pd.to_numeric(master["__label__"], errors="coerce").fillna(0.0)

# Normalize obviously negative labels to nonnegative values
if master["__label__"].min() < 0:
    master["__label__"] = master["__label__"] - master["__label__"].min()

print(master["__label__"].describe())
print("Positive relevance rate:", (master["__label__"] > 0).mean())


In [ ]:

# =========================
# 7. DEFINE COLD-START PRODUCTS
# =========================
history_cols = [c for c in master.columns if c in HISTORY_EXACT]

definition = {
    "type": None,
    "history_columns": history_cols,
    "positive_quantile": COLD_START_POSITIVE_QUANTILE,
    "simulated_fraction": SIMULATED_COLD_START_FRACTION,
    "rule": None,
}

product_frame = master[["__product__"]].drop_duplicates().copy()

if history_cols:
    # Aggregate product-level history. "max" is robust when the count is repeated per query-product row.
    tmp = master[["__product__"] + history_cols].copy()
    for c in history_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce").fillna(0).clip(lower=0)
    prod_hist = tmp.groupby("__product__", as_index=False)[history_cols].max()

    # Composite history score:
    # log1p each count, scale by its 95th percentile, average across available signals.
    scaled = []
    for c in history_cols:
        x = np.log1p(prod_hist[c].astype(float))
        denom = np.nanpercentile(x, 95)
        if not np.isfinite(denom) or denom <= 0:
            denom = 1.0
        scaled.append(np.clip(x / denom, 0, 1))
    prod_hist["__history_score__"] = np.mean(np.vstack(scaled), axis=0)

    raw_zero = (prod_hist[history_cols].sum(axis=1) == 0)
    positive_scores = prod_hist.loc[~raw_zero, "__history_score__"]

    if len(positive_scores) > 0:
        threshold = float(positive_scores.quantile(COLD_START_POSITIVE_QUANTILE))
    else:
        threshold = 0.0

    prod_hist["__cold_start__"] = raw_zero | (prod_hist["__history_score__"] <= threshold)

    # Avoid pathological all/none splits
    share = prod_hist["__cold_start__"].mean()
    if share < 0.05 or share > 0.60:
        threshold = float(prod_hist["__history_score__"].quantile(0.20))
        prod_hist["__cold_start__"] = prod_hist["__history_score__"] <= threshold

    product_frame = prod_hist
    definition["type"] = "observed_history"
    definition["threshold"] = threshold
    definition["rule"] = (
        "Cold-start = product has zero total observed history OR composite normalized history "
        f"score is at/below the bottom {int(COLD_START_POSITIVE_QUANTILE*100)}% of positive-history products."
    )

else:
    # Honest fallback: deterministic unseen-product simulation.
    def stable_unit_hash(x):
        h = hashlib.sha1(str(x).encode("utf-8")).hexdigest()
        return int(h[:12], 16) / float(16**12 - 1)

    product_frame["__hash_u__"] = product_frame["__product__"].map(stable_unit_hash)
    product_frame["__cold_start__"] = product_frame["__hash_u__"] < SIMULATED_COLD_START_FRACTION
    product_frame["__history_score__"] = np.nan

    definition["type"] = "simulated_unseen_product_holdout"
    definition["threshold"] = SIMULATED_COLD_START_FRACTION
    definition["rule"] = (
        f"No genuine interaction-count fields were detected. Cold-start is simulated as a deterministic "
        f"{int(SIMULATED_COLD_START_FRACTION*100)}% product-level unseen holdout based on SHA-1 hash. "
        "This is an unseen-product generalization test, not a claim about true click/review scarcity."
    )

master = master.merge(
    product_frame[["__product__", "__cold_start__", "__history_score__"]],
    on="__product__",
    how="left"
)

cold = master[master["__cold_start__"] == True].copy()
hist = master[master["__cold_start__"] == False].copy()

definition.update({
    "n_products_total": int(master["__product__"].nunique()),
    "n_products_cold_start": int(cold["__product__"].nunique()),
    "cold_start_product_share": float(cold["__product__"].nunique() / max(master["__product__"].nunique(), 1)),
    "n_pairs_total": int(len(master)),
    "n_pairs_cold_start": int(len(cold)),
    "n_queries_cold_start": int(cold["__query__"].nunique()),
})

print(json.dumps(definition, indent=2))
print("\nCold-start pair table:", cold.shape)
display(cold.head())


In [ ]:

# =========================
# 8. DETECT RANKING SCORE COLUMNS
# =========================
score_map = defaultdict(list)
for c in master.columns:
    fam = score_family(c)
    if fam:
        score_map[fam].append(c)

# Prefer one numeric score per family with the highest non-null coverage
selected_scores = {}
for fam, cols in score_map.items():
    ranked = []
    for c in cols:
        x = pd.to_numeric(master[c], errors="coerce")
        coverage = x.notna().mean()
        nunique = x.nunique(dropna=True)
        ranked.append((coverage, nunique, c))
    ranked.sort(reverse=True)
    if ranked:
        selected_scores[fam] = ranked[0][2]

print("Selected model score columns:")
for model, col in selected_scores.items():
    print(f" - {model}: {col} (coverage={master[col].notna().mean():.1%})")

if not selected_scores:
    raise RuntimeError(
        "No BM25/SBERT/CLIP/LambdaMART/ListNet score columns were detected. "
        "Upload Week 2/4/5 pair-level result files or the Week 5 feature dataset."
    )


In [ ]:

# =========================
# 9. RANKING METRICS
# =========================
def dcg_at_k(rels, k=10):
    rels = np.asarray(rels, dtype=float)[:k]
    if len(rels) == 0:
        return np.nan
    discounts = np.log2(np.arange(2, len(rels) + 2))
    gains = (2 ** rels - 1)
    return float(np.sum(gains / discounts))

def ndcg_at_k(y_true, y_score, k=10):
    order = np.argsort(-np.asarray(y_score, dtype=float))
    ranked_rels = np.asarray(y_true, dtype=float)[order]
    ideal_rels = np.sort(np.asarray(y_true, dtype=float))[::-1]
    denom = dcg_at_k(ideal_rels, k)
    if not np.isfinite(denom) or denom <= 0:
        return np.nan
    return dcg_at_k(ranked_rels, k) / denom

def recall_at_k(y_true, y_score, k=10):
    y_true = np.asarray(y_true, dtype=float)
    relevant = y_true > 0
    total_rel = relevant.sum()
    if total_rel == 0:
        return np.nan
    order = np.argsort(-np.asarray(y_score, dtype=float))[:k]
    return float(relevant[order].sum() / total_rel)

def mrr_score(y_true, y_score):
    y_true = np.asarray(y_true, dtype=float)
    order = np.argsort(-np.asarray(y_score, dtype=float))
    ranked = y_true[order]
    hits = np.where(ranked > 0)[0]
    if len(hits) == 0:
        return 0.0
    return float(1.0 / (hits[0] + 1))

def evaluate_subset(df, score_col, model_name, subset_name, k=10):
    rows = []
    working = df[["__query__", "__label__", score_col]].copy()
    working[score_col] = pd.to_numeric(working[score_col], errors="coerce")
    working = working.dropna(subset=[score_col])

    for q, g in working.groupby("__query__"):
        # Need at least 2 candidates for a meaningful ranking query
        if len(g) < 2:
            continue
        y = g["__label__"].to_numpy(float)
        s = g[score_col].to_numpy(float)
        rows.append({
            "subset": subset_name,
            "model": model_name,
            "query": q,
            "n_candidates": len(g),
            f"ndcg@{k}": ndcg_at_k(y, s, k),
            f"recall@{k}": recall_at_k(y, s, k),
            "mrr": mrr_score(y, s)
        })

    per_q = pd.DataFrame(rows)
    if per_q.empty:
        summary = {
            "subset": subset_name, "model": model_name,
            "n_queries": 0, f"ndcg@{k}": np.nan, f"recall@{k}": np.nan, "mrr": np.nan
        }
    else:
        summary = {
            "subset": subset_name,
            "model": model_name,
            "n_queries": int(len(per_q)),
            f"ndcg@{k}": float(per_q[f"ndcg@{k}"].mean(skipna=True)),
            f"recall@{k}": float(per_q[f"recall@{k}"].mean(skipna=True)),
            "mrr": float(per_q["mrr"].mean(skipna=True)),
        }
    return summary, per_q

summaries = []
per_query_parts = []

for subset_name, subset_df in [("cold_start", cold), ("historical", hist)]:
    for model, col in selected_scores.items():
        summary, pq = evaluate_subset(subset_df, col, model, subset_name, K)
        summaries.append(summary)
        if not pq.empty:
            per_query_parts.append(pq)

metrics = pd.DataFrame(summaries)
per_query = pd.concat(per_query_parts, ignore_index=True) if per_query_parts else pd.DataFrame()

cold_metrics = metrics[metrics["subset"] == "cold_start"].copy()
hist_metrics = metrics[metrics["subset"] == "historical"].copy()

print("COLD-START BENCHMARK")
display(cold_metrics.sort_values(f"ndcg@{K}", ascending=False))

print("\nHISTORICAL BENCHMARK")
display(hist_metrics.sort_values(f"ndcg@{K}", ascending=False))


In [ ]:

# =========================
# 10. RELATIVE IMPROVEMENT VS BM25
# =========================
metric_col = f"ndcg@{K}"

cold_metrics["ndcg_improvement_vs_bm25_pct"] = np.nan

if "BM25" in set(cold_metrics["model"]):
    bm25_val = cold_metrics.loc[cold_metrics["model"] == "BM25", metric_col].iloc[0]
    if pd.notna(bm25_val) and bm25_val != 0:
        cold_metrics["ndcg_improvement_vs_bm25_pct"] = (
            (cold_metrics[metric_col] - bm25_val) / bm25_val * 100
        )

display(cold_metrics.sort_values(metric_col, ascending=False))

best_row = cold_metrics.sort_values(metric_col, ascending=False).head(1)
if len(best_row):
    best_model = best_row.iloc[0]["model"]
    best_ndcg = best_row.iloc[0][metric_col]
    print(f"Best cold-start model: {best_model} | NDCG@{K}={best_ndcg:.4f}")


In [ ]:

# =========================
# 11. DIAGNOSTIC PLOTS
# =========================
# Plot 1: Cold-start NDCG@10
plot_df = cold_metrics.dropna(subset=[f"ndcg@{K}"]).sort_values(f"ndcg@{K}", ascending=False)

if not plot_df.empty:
    plt.figure(figsize=(8, 5))
    plt.bar(plot_df["model"], plot_df[f"ndcg@{K}"])
    plt.ylabel(f"NDCG@{K}")
    plt.xlabel("Model")
    plt.title("Week 6 V1 — Cold-Start Ranking Benchmark")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "cold_start_ndcg10.png", dpi=180, bbox_inches="tight")
    plt.show()

# Plot 2: Cold-start vs Historical NDCG@10
pivot = metrics.pivot_table(index="model", columns="subset", values=f"ndcg@{K}", aggfunc="first").dropna(how="all")
if not pivot.empty:
    pivot.plot(kind="bar", figsize=(9, 5))
    plt.ylabel(f"NDCG@{K}")
    plt.xlabel("Model")
    plt.title("Cold-Start vs Historical Ranking Quality")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "cold_start_vs_historical_ndcg10.png", dpi=180, bbox_inches="tight")
    plt.show()

# Plot 3: Product split
split_counts = pd.Series({
    "Cold-start": cold["__product__"].nunique(),
    "Historical": hist["__product__"].nunique()
})
plt.figure(figsize=(6, 4))
plt.bar(split_counts.index, split_counts.values)
plt.ylabel("Unique products")
plt.title("Week 6 V1 — Product Split")
plt.tight_layout()
plt.savefig(OUT_DIR / "cold_start_product_split.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

# =========================
# 12. SAVE V1 OUTPUTS
# =========================
cold.to_csv(OUT_DIR / "cold_start_subset.csv", index=False)
product_frame.to_csv(OUT_DIR / "product_cold_start_flags.csv", index=False)

cold_metrics.to_csv(OUT_DIR / "cold_start_benchmark_metrics.csv", index=False)
hist_metrics.to_csv(OUT_DIR / "historical_benchmark_metrics.csv", index=False)
metrics.to_csv(OUT_DIR / "all_subset_benchmark_metrics.csv", index=False)
if not per_query.empty:
    per_query.to_csv(OUT_DIR / "cold_start_per_query_metrics.csv", index=False)

with open(OUT_DIR / "cold_start_definition.json", "w") as f:
    json.dump(definition, f, indent=2)

# Build readable summary
lines = []
lines.append("Amazon Search Ranking — Week 6 V1")
lines.append("Cold-Start Subset Definition + Baseline Benchmark")
lines.append("=" * 64)
lines.append("")
lines.append(f"Cold-start definition type: {definition['type']}")
lines.append(f"Definition rule: {definition['rule']}")
lines.append(f"Total products: {definition['n_products_total']:,}")
lines.append(f"Cold-start products: {definition['n_products_cold_start']:,}")
lines.append(f"Cold-start share: {definition['cold_start_product_share']:.2%}")
lines.append(f"Cold-start query-product pairs: {definition['n_pairs_cold_start']:,}")
lines.append(f"Cold-start queries: {definition['n_queries_cold_start']:,}")
lines.append("")
lines.append("Selected score columns:")
for model, col in selected_scores.items():
    lines.append(f"  - {model}: {col}")
lines.append("")
lines.append("Cold-start benchmark:")
if cold_metrics.empty:
    lines.append("  No valid model metrics were produced.")
else:
    for _, r in cold_metrics.sort_values(f"ndcg@{K}", ascending=False).iterrows():
        imp = r.get("ndcg_improvement_vs_bm25_pct", np.nan)
        imp_txt = "n/a" if pd.isna(imp) else f"{imp:+.2f}%"
        lines.append(
            f"  - {r['model']}: NDCG@{K}={r[f'ndcg@{K}']:.4f}, "
            f"Recall@{K}={r[f'recall@{K}']:.4f}, MRR={r['mrr']:.4f}, "
            f"vs BM25={imp_txt}"
        )

if len(best_row):
    lines.append("")
    lines.append(f"Best cold-start model: {best_model}")
    lines.append(f"Best cold-start NDCG@{K}: {best_ndcg:.4f}")
    if "BM25" in set(cold_metrics["model"]):
        imp = best_row.iloc[0].get("ndcg_improvement_vs_bm25_pct", np.nan)
        if pd.notna(imp):
            lines.append(f"Improvement over BM25: {imp:+.2f}%")
            lines.append(f"15% target achieved: {'YES' if imp >= 15 else 'NO'}")

summary_text = "\n".join(lines)
(OUT_DIR / "cold_start_summary.txt").write_text(summary_text)

print(summary_text)


In [ ]:

# =========================
# 13. PACKAGE + DOWNLOAD
# =========================
zip_path = ROOT / "amazon_week6_v1_coldstart_benchmark_results.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT_DIR))

print("Created:", zip_path)
print("Size:", f"{zip_path.stat().st_size / 1024 / 1024:.2f} MB")

from google.colab import files
files.download(str(zip_path))



## How to interpret V1

### If `definition.type = observed_history`
Your uploaded data contained genuine history-like count fields. The cold-start subset is based on zero / low historical activity.

### If `definition.type = simulated_unseen_product_holdout`
The available Amazon dataset did not contain a genuine behavioral history field. The notebook therefore performs an honest product-level unseen-item generalization test instead of mislabeling relevance data as clicks/reviews.

### Decision for V2
Use the V1 benchmark to identify the strongest non-history baseline. In V2, we will build a **zero-shot ranking adapter** that emphasizes metadata / semantic / multimodal signals and compare it against this V1 benchmark.
